# Phân tích cảm xúc đánh giá phim IMDB: RNN (BiGRU) so với Mạng nơ-ron cơ bản và Học máy truyền thống

**Bộ dữ liệu:** [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) (Kaggle), gồm 50.000 bài đánh giá phim được gán nhãn `positive` (tích cực) / `negative` (tiêu cực), mỗi loại 25.000 bài.

**Mục tiêu:** dùng mạng nơ-ron hồi quy (RNN) để phân loại cảm xúc của bài đánh giá, sau đó so sánh với
1. **Học máy truyền thống:** Logistic Regression, Linear SVM và Multinomial Naive Bayes trên đặc trưng TF-IDF
2. **Mạng nơ-ron cơ bản (truyền thẳng):** một MLP trên đặc trưng TF-IDF, và một mạng EmbeddingBag (lấy trung bình word embedding + lớp dense)
3. **Mạng nơ-ron hồi quy:** một mạng GRU hai chiều (BiGRU) huấn luyện trên chuỗi chỉ số từ

Mọi mô hình đều dùng **cùng một cách chia train/validation/test**, và mọi kết quả đều được báo cáo trên tập test giữ riêng.

## Mục lục
1. Cài đặt & nạp dữ liệu
2. Phân tích khám phá dữ liệu (EDA)
3. Làm sạch văn bản & chia tập train/val/test
4. Học máy truyền thống (TF-IDF + LR / SVM / NB)
5. Mạng nơ-ron cơ bản (MLP, EmbeddingBag)
6. Mạng nơ-ron hồi quy (BiGRU)
7. So sánh & thảo luận

> **Cách chạy:** cần Python 3.10+ với các thư viện `numpy pandas scikit-learn torch matplotlib seaborn`.
> Đặt file `IMDB Dataset.csv` (tải từ trang Kaggle ở trên) cùng thư mục với notebook, rồi chạy toàn bộ các ô (Run All).
> Notebook tự động dùng GPU NVIDIA nếu PyTorch nhận được GPU; khi đó toàn bộ notebook chạy trong khoảng 5–10 phút.

## 1. Cài đặt & nạp dữ liệu

In [ ]:
import os, re, time, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count() or 1)
sns.set_theme(style="whitegrid")
print("torch", torch.__version__, "| thiết bị:", DEVICE)

Notebook tìm bộ dữ liệu theo thứ tự sau:
1. `IMDB Dataset.csv` hoặc `data/IMDB Dataset.csv` (tải file zip từ trang Kaggle rồi giải nén)
2. đường dẫn input của Kaggle Notebook: `/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv`
3. `kagglehub.dataset_download(...)`, cần có thông tin xác thực Kaggle API

In [ ]:
def find_dataset():
    candidates = [Path("IMDB Dataset.csv"), Path("data/IMDB Dataset.csv"),
                  Path("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")]
    for p in candidates:
        if p.exists():
            return p
    import kagglehub  # cài bằng: pip install kagglehub
    return Path(kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")) / "IMDB Dataset.csv"

df = pd.read_csv(find_dataset())
print(df.shape)
df.head()

## 2. Phân tích khám phá dữ liệu (EDA)

In [ ]:
print(df["sentiment"].value_counts())
print("Số giá trị thiếu:", df.isna().sum().sum())
print("Số bài đánh giá trùng lặp:", df["review"].duplicated().sum())

df["n_words"] = df["review"].str.split().str.len()
print(df["n_words"].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["sentiment"].value_counts().plot.bar(ax=axes[0], color=["#4c72b0", "#dd8452"], rot=0)
axes[0].set_title("Phân bố nhãn")
sns.histplot(data=df[df.n_words < 1200], x="n_words", hue="sentiment", bins=60, ax=axes[1])
axes[1].axvline(256, color="k", ls="--", lw=1, label="Độ dài tối đa cho RNN (256)")
axes[1].set(title="Độ dài bài đánh giá (số từ)", xlabel="số từ", ylabel="số bài")
plt.tight_layout(); plt.show()

Hai lớp cân bằng hoàn toàn, nên **độ chính xác (accuracy)** là thước đo chính hợp lý. Các bài đánh giá khá dài (trung vị ≈ 174 từ, có đuôi dài).
Điều này quan trọng với RNN: mô hình phải mang thông tin qua hàng trăm bước thời gian.
Bộ dữ liệu có 418 bài đánh giá bị trùng lặp. Chúng tôi loại bỏ chúng **trước khi** chia tập, để cùng một văn bản không xuất hiện ở cả tập train lẫn tập test.

## 3. Làm sạch văn bản & chia tập train/val/test

In [ ]:
TAG_RE = re.compile(r"<[^>]+>")
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[!?]")

def clean(text):
    text = TAG_RE.sub(" ", text.lower())          # bỏ thẻ HTML như "<br />"
    return " ".join(TOKEN_RE.findall(text))       # giữ lại từ, từ rút gọn, dấu ! và ?

df = df.drop_duplicates("review").reset_index(drop=True)
df["text"] = df["review"].map(clean)
df["label"] = (df["sentiment"] == "positive").astype(int)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df.label, random_state=SEED)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
print("\nVí dụ văn bản sau khi làm sạch:\n", train_df.text.iloc[0][:300])

y_train, y_val, y_test = train_df.label.values, val_df.label.values, test_df.label.values

Các từ dừng (stop-words) được **giữ lại** có chủ đích. Những từ như *not*, *no*, *but* có thể đảo ngược cảm xúc, và mô hình chuỗi có thể tận dụng chúng.

Tỷ lệ chia là 80 / 10 / 10. Tập validation dùng cho early stopping và chọn tham số, còn tập test chỉ được dùng cho lần đánh giá cuối cùng.

### Hàm đánh giá dùng chung

In [ ]:
TRAD, BASIC, RNN_FAMILY = "Học máy truyền thống", "Mạng nơ-ron cơ bản", "RNN"
results = {}       # tên mô hình -> dict các chỉ số
predictions = {}   # tên mô hình -> (y_pred, y_score)

def evaluate(name, y_true, y_pred, y_score, train_time, infer_time, n_params=None, family=""):
    m = dict(family=family,
             accuracy=accuracy_score(y_true, y_pred),
             precision=precision_score(y_true, y_pred),
             recall=recall_score(y_true, y_pred),
             f1=f1_score(y_true, y_pred),
             roc_auc=roc_auc_score(y_true, y_score),
             train_time_s=train_time,
             infer_time_s=infer_time,
             params=n_params)
    results[name] = m
    predictions[name] = (y_pred, y_score)
    print(f"{name:<22} acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  auc={m['roc_auc']:.4f}  "
          f"thời gian huấn luyện={train_time:.1f}s")
    return m

## 4. Học máy truyền thống: TF-IDF + mô hình tuyến tính

Mỗi bài đánh giá được biểu diễn thành một vector TF-IDF thưa trên unigram và bigram. Bigram nắm bắt được các cụm ngắn như *"not good"* hay *"waste of"*,
điều mà bag-of-words thuần túy bỏ lỡ. Các mô hình này bỏ qua thứ tự từ, ngoài phạm vi cửa sổ hai từ.

In [ ]:
t0 = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=200_000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df.text)
X_val_tfidf = tfidf.transform(val_df.text)
X_test_tfidf = tfidf.transform(test_df.text)
print(f"Ma trận TF-IDF: {X_train_tfidf.shape}, xây dựng trong {time.time() - t0:.1f}s")

Một lưới nhỏ các giá trị cho hệ số điều chuẩn `C` (hoặc `alpha` với NB) được tìm kiếm trên tập **validation**.

In [ ]:
def tune_and_eval(name, make_model, grid, score_fn):
    best = None
    for p in grid:
        model = make_model(p).fit(X_train_tfidf, y_train)
        acc = accuracy_score(y_val, model.predict(X_val_tfidf))
        print(f"  {name} tham số={p:<6} val_acc={acc:.4f}")
        if best is None or acc > best[0]:
            best = (acc, p)
    # huấn luyện lại với tham số tốt nhất trên tập train và đo thời gian
    t0 = time.time(); model = make_model(best[1]).fit(X_train_tfidf, y_train); train_t = time.time() - t0
    t0 = time.time(); y_pred = model.predict(X_test_tfidf); score = score_fn(model, X_test_tfidf); inf_t = time.time() - t0
    n_params = int(np.prod(model.coef_.shape)) if hasattr(model, "coef_") else int(np.prod(model.feature_log_prob_.shape))
    evaluate(name, y_test, y_pred, score, train_t, inf_t, n_params, family=TRAD)
    return model

logreg = tune_and_eval("Logistic Regression",
                       lambda C: LogisticRegression(C=C, max_iter=2000, solver="liblinear"),
                       [1, 4, 10, 30], lambda m, X: m.predict_proba(X)[:, 1])
svm = tune_and_eval("Linear SVM",
                    lambda C: LinearSVC(C=C),
                    [0.1, 0.3, 1, 3], lambda m, X: m.decision_function(X))
nb = tune_and_eval("Naive Bayes",
                   lambda a: MultinomialNB(alpha=a),
                   [0.03, 0.1, 0.3, 1.0], lambda m, X: m.predict_proba(X)[:, 1])

Vì Logistic Regression là mô hình tuyến tính, ta có thể đọc trực tiếp những n-gram nào đẩy dự đoán về phía tích cực hoặc tiêu cực:

In [ ]:
feat = np.array(tfidf.get_feature_names_out())
coef = logreg.coef_[0]
pd.DataFrame({"tiêu cực nhất": feat[np.argsort(coef)[:15]],
              "tích cực nhất": feat[np.argsort(coef)[::-1][:15]]})

## 5. Mạng nơ-ron cơ bản (truyền thẳng)

Hai mạng không hồi quy:

* **MLP trên TF-IDF.** Dùng cùng loại đặc trưng thưa như các mô hình tuyến tính (20k unigram + bigram phổ biến nhất), đi qua `Linear(20k→256) → ReLU → Dropout → Linear(256→1)`.
  Đây là "mạng nơ-ron cơ bản" theo nghĩa kinh điển: mạng kết nối đầy đủ với một lớp ẩn.
* **EmbeddingBag (trung bình word embedding).** Mô hình học một embedding 128 chiều cho mỗi từ, **lấy trung bình** các embedding trên toàn bài đánh giá, rồi đưa qua một đầu dense nhỏ.
  Mạng này nhận cùng chuỗi token như RNN nhưng hoàn toàn bỏ qua thứ tự từ, nên là mốc so sánh tự nhiên cho RNN.

### Vòng lặp huấn luyện PyTorch dùng chung
Mọi mạng đều dùng hàm mất mát binary cross-entropy (trên logit), bộ tối ưu Adam, gradient clipping, và **early stopping theo accuracy trên tập validation**: giữ lại trọng số của epoch tốt nhất.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def predict_scores(model, loader):
    model.eval()
    out = []
    for batch in loader:
        *inputs, _ = [b.to(DEVICE) for b in batch]
        out.append(torch.sigmoid(model(*inputs)).cpu())
    return torch.cat(out).numpy()

def train_model(name, model, train_loader, val_loader, test_loader, epochs=10, lr=1e-3,
                weight_decay=0.0, patience=2, clip=1.0, family=""):
    set_seed()
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    history, best_acc, best_state, bad = [], -1, None, 0
    t_start = time.time()
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot, n, correct = 0.0, 0, 0
        for batch in train_loader:
            *inputs, y = [b.to(DEVICE) for b in batch]
            opt.zero_grad()
            logits = model(*inputs)
            loss = loss_fn(logits, y.float())
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            tot += loss.item() * len(y); n += len(y); correct += ((logits > 0).long() == y).sum().item()
        val_scores = predict_scores(model, val_loader)
        val_loss = loss_fn(torch.logit(torch.tensor(val_scores).clamp(1e-7, 1 - 1e-7)), torch.tensor(y_val).float()).item()
        val_acc = accuracy_score(y_val, val_scores > 0.5)
        history.append(dict(epoch=epoch, train_loss=tot / n, train_acc=correct / n, val_loss=val_loss, val_acc=val_acc))
        print(f"[{name}] epoch {epoch:2d}  train_loss={tot/n:.4f} train_acc={correct/n:.4f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}  ({time.time()-t0:.0f}s)")
        if val_acc > best_acc:
            best_acc, bad = val_acc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"[{name}] dừng sớm (early stopping); val_acc tốt nhất={best_acc:.4f}")
                break
    train_time = time.time() - t_start
    model.load_state_dict(best_state)
    t0 = time.time(); scores = predict_scores(model, test_loader); infer_time = time.time() - t0
    evaluate(name, y_test, (scores > 0.5).astype(int), scores, train_time, infer_time,
             count_params(model), family=family)
    histories[name] = pd.DataFrame(history)
    return model

histories = {}

### 5.1 MLP trên đặc trưng TF-IDF

In [ ]:
tfidf_nn = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=20_000, sublinear_tf=True)
Xtr_nn = tfidf_nn.fit_transform(train_df.text).astype(np.float32)
Xva_nn = tfidf_nn.transform(val_df.text).astype(np.float32)
Xte_nn = tfidf_nn.transform(test_df.text).astype(np.float32)

class SparseDataset(Dataset):
    # giữ ma trận ở dạng thưa; các hàng chỉ được chuyển sang dạng dày theo từng batch trong collate fn
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return i

def sparse_loader(X, y, shuffle, bs=256):
    def collate(idx):
        return torch.from_numpy(X[idx].toarray()), torch.from_numpy(y[idx])
    return DataLoader(SparseDataset(X, y), batch_size=bs, shuffle=shuffle, collate_fn=collate)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=256, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

mlp = train_model("MLP (TF-IDF)", MLP(Xtr_nn.shape[1]),
                  sparse_loader(Xtr_nn, y_train, True), sparse_loader(Xva_nn, y_val, False),
                  sparse_loader(Xte_nn, y_test, False), epochs=10, lr=1e-3, weight_decay=1e-5,
                  family=BASIC)

### Tách từ (tokenisation) cho các mô hình chuỗi
Mô hình EmbeddingBag và BiGRU dùng chung bộ từ vựng gồm 30k từ xuất hiện nhiều nhất trong tập train (chỉ số 0 = padding, 1 = từ không xác định).
Mỗi bài đánh giá được giới hạn ở **256 token**, đủ để bao trọn toàn bộ bài với khoảng 70% dữ liệu. Với bài dài hơn, ta giữ **256 token cuối cùng**,
vì người viết thường nêu nhận xét tổng kết ở cuối bài. Các chuỗi được đệm (padding) theo từng batch, và độ dài thực được truyền vào RNN (`pack_padded_sequence`) để phần đệm không ảnh hưởng tới trạng thái hồi quy.

In [ ]:
MAX_LEN, VOCAB_SIZE = 256, 30_000
counter = Counter(tok for text in train_df.text for tok in text.split())
itos = ["<pad>", "<unk>"] + [w for w, _ in counter.most_common(VOCAB_SIZE - 2)]
stoi = {w: i for i, w in enumerate(itos)}

def encode(text):
    ids = [stoi.get(t, 1) for t in text.split()]
    return ids[-MAX_LEN:] if ids else [1]

seq_train = [encode(t) for t in train_df.text]
seq_val = [encode(t) for t in val_df.text]
seq_test = [encode(t) for t in test_df.text]
lens = np.array([len(s) for s in seq_train])
print(f"từ vựng={len(itos)}  tỷ lệ token train được bao phủ={sum(counter[w] for w in itos[2:]) / sum(counter.values()):.3f}  "
      f"bài không bị cắt={np.mean(lens < MAX_LEN):.2%}")

class SeqDataset(Dataset):
    def __init__(self, seqs, y): self.seqs, self.y = seqs, y
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i): return self.seqs[i], self.y[i]

def pad_collate(batch):
    seqs, ys = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = torch.zeros(len(seqs), lengths.max(), dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = torch.tensor(s)
    return padded, lengths, torch.tensor(ys)

def seq_loader(seqs, y, shuffle, bs=64):
    return DataLoader(SeqDataset(seqs, y), batch_size=bs, shuffle=shuffle, collate_fn=pad_collate)

train_loader = seq_loader(seq_train, y_train, True)
val_loader, test_loader = seq_loader(seq_val, y_val, False), seq_loader(seq_test, y_test, False)

### 5.2 EmbeddingBag: trung bình word embedding + lớp dense (không quan tâm thứ tự từ)

In [ ]:
class EmbeddingBagNet(nn.Module):
    def __init__(self, vocab, emb=128, hidden=64, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(emb, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x, lengths):
        mean = self.emb(x).sum(1) / lengths.unsqueeze(1).to(x.device)   # embedding của padding bằng 0
        return self.head(mean).squeeze(-1)

embbag = train_model("EmbeddingBag NN", EmbeddingBagNet(len(itos)), train_loader, val_loader, test_loader,
                     epochs=15, lr=1e-3, patience=3, family=BASIC)

## 6. Mạng nơ-ron hồi quy: GRU hai chiều (BiGRU)

Kiến trúc:

`Embedding(30k, 128) → GRU hai chiều, 64 đơn vị ẩn mỗi chiều (packed sequence) → nối trạng thái ẩn cuối của hai chiều → Dropout → Linear → logit`

**Vì sao chọn GRU thay vì RNN cơ bản?** Với RNN cơ bản (`tanh`), thông tin và gradient phải đi qua tới 256 phép nhân ma trận lặp lại, nên gradient bị **tiêu biến (vanishing gradient)** trước khi tới được các token đầu của bài.
GRU có hai cổng (update gate và reset gate) quyết định giữ lại bao nhiêu trạng thái cũ, tạo ra một đường truyền dạng cộng qua thời gian, nhờ đó gradient không bị tiêu biến.
So với LSTM, GRU có ít tham số hơn (không có cell state riêng) và thường hội tụ nhanh hơn.

**Vì sao hai chiều?** Mạng đọc bài đánh giá cả từ đầu đến cuối và từ cuối về đầu, nên cả phần mở đầu lẫn phần kết của bài đều "gần" với trạng thái cuối cùng.

Vì độ dài thực được truyền qua `pack_padded_sequence`, "trạng thái ẩn cuối" là trạng thái tại **token thực cuối cùng** của mỗi bài, chứ không phải trạng thái sau phần đệm.

In [ ]:
class BiGRUClassifier(nn.Module):
    def __init__(self, vocab, emb=128, hidden=64, dropout=0.4):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.gru = nn.GRU(emb, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden, 1)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)                  # h: (2, batch, hidden), trạng thái cuối của mỗi chiều
        h = torch.cat([h[0], h[1]], dim=1)       # nối chiều xuôi và chiều ngược
        return self.fc(self.drop(h)).squeeze(-1)

bigru = train_model("BiGRU", BiGRUClassifier(len(itos)), train_loader, val_loader, test_loader,
                    epochs=15, lr=1e-3, patience=3, family=RNN_FAMILY)

### Đường cong huấn luyện

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in histories.items():
    line, = axes[0].plot(h.epoch, h.train_loss, "--", alpha=0.6)
    axes[0].plot(h.epoch, h.val_loss, "-o", color=line.get_color(), ms=3, label=name)
    axes[1].plot(h.epoch, h.val_acc, "-o", ms=3, label=name)
axes[0].set(title="Loss (nét đứt = train, nét liền = validation)", xlabel="epoch", ylabel="BCE loss")
axes[1].set(title="Accuracy trên tập validation", xlabel="epoch", ylabel="accuracy")
axes[1].legend()
plt.tight_layout(); plt.show()

## 7. So sánh

In [ ]:
res = pd.DataFrame(results).T
res.index.name = "mô hình"
res = res.sort_values("accuracy", ascending=False)
fmt = {c: "{:.4f}" for c in ["accuracy", "precision", "recall", "f1", "roc_auc"]}
fmt.update(train_time_s="{:.1f}", infer_time_s="{:.2f}", params="{:,.0f}")
res.style.format(fmt).background_gradient(subset=["accuracy", "f1", "roc_auc"], cmap="Greens")

In [ ]:
palette = {TRAD: "#4c72b0", BASIC: "#dd8452", RNN_FAMILY: "#55a868"}
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
order = res.sort_values("accuracy")
axes[0].barh(order.index, order.accuracy, color=order.family.map(palette))
for i, v in enumerate(order.accuracy):
    axes[0].text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)
axes[0].set_xlim(order.accuracy.min() - 0.03, order.accuracy.max() + 0.015)
axes[0].set_title("Accuracy trên tập test")
axes[1].scatter(res.train_time_s, res.accuracy, c=res.family.map(palette), s=80)
for name, r in res.iterrows():
    axes[1].annotate(name, (r.train_time_s, r.accuracy), textcoords="offset points", xytext=(6, -3), fontsize=9)
axes[1].set_xscale("log")
axes[1].set(title="Accuracy so với thời gian huấn luyện", xlabel="thời gian huấn luyện [giây, thang log]", ylabel="accuracy (test)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in palette.values()]
axes[0].legend(handles, palette.keys(), loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
names = list(res.index)
fig, axes = plt.subplots(2, (len(names) + 1) // 2, figsize=(4 * ((len(names) + 1) // 2), 7.5))
for ax, name in zip(axes.flat, names):
    cm = confusion_matrix(y_test, predictions[name][0])
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["tiêu cực", "tích cực"], yticklabels=["tiêu cực", "tích cực"])
    ax.set(title=f"{name}\nacc={res.loc[name, 'accuracy']:.4f}", xlabel="dự đoán", ylabel="thực tế")
for ax in axes.flat[len(names):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### Độ dài bài đánh giá có ảnh hưởng không?
Chia tập test thành các nhóm theo độ dài và so sánh mô hình tốt nhất của mỗi họ. Bài dài là nơi BiGRU (mô hình có xét thứ tự từ) lẽ ra phát huy ưu thế,
nhưng cũng là nơi việc cắt còn 256 token gây bất lợi cho nó.

In [ ]:
test_len = test_df.text.str.split().str.len().values
buckets = pd.cut(test_len, [0, 100, 200, 400, 10_000], labels=["≤100", "101-200", "201-400", ">400"])
best_per_family = res.groupby("family")["accuracy"].idxmax().tolist()
by_len = pd.DataFrame({name: pd.Series(predictions[name][0] == y_test).groupby(buckets, observed=True).mean()
                       for name in best_per_family})
by_len["số bài"] = pd.Series(buckets).value_counts().reindex(by_len.index).values
by_len.index.name = "số từ"
by_len.round(4)

### Các mô hình bất đồng ở đâu?

In [ ]:
best_trad = res[res.family == TRAD].index[0]
p_rnn, p_trad = predictions["BiGRU"][0], predictions[best_trad][0]
print(f"BiGRU đúng & {best_trad} sai: {np.sum((p_rnn == y_test) & (p_trad != y_test))}")
print(f"{best_trad} đúng & BiGRU sai: {np.sum((p_trad == y_test) & (p_rnn != y_test))}")
print(f"cả hai cùng sai: {np.sum((p_rnn != y_test) & (p_trad != y_test))} / {len(y_test)}")

# ensemble đơn giản: trung bình xác suất của hai mô hình
p_ens = (predictions["Logistic Regression"][1] + predictions["BiGRU"][1]) / 2
print(f"\nAccuracy của ensemble (Logistic Regression + BiGRU): {accuracy_score(y_test, p_ens > 0.5):.4f}")

### Thử trên một vài câu tự viết (giá trị = xác suất tích cực)

In [ ]:
def predict_all(texts):
    cleaned = [clean(t) for t in texts]
    rows = {}
    Xt = tfidf.transform(cleaned)
    rows["Logistic Regression"] = logreg.predict_proba(Xt)[:, 1]
    rows["Linear SVM"] = 1 / (1 + np.exp(-svm.decision_function(Xt)))     # nén margin về (0, 1), chỉ để hiển thị
    rows["Naive Bayes"] = nb.predict_proba(Xt)[:, 1]
    rows["MLP (TF-IDF)"] = torch.sigmoid(mlp(torch.from_numpy(tfidf_nn.transform(cleaned).astype(np.float32).toarray()).to(DEVICE))).detach().cpu().numpy()
    seqs = [encode(t) for t in cleaned]
    x, lengths, _ = pad_collate(list(zip(seqs, [0] * len(seqs))))
    for name, m in [("EmbeddingBag NN", embbag), ("BiGRU", bigru)]:
        m.eval()
        with torch.no_grad():
            rows[name] = torch.sigmoid(m(x.to(DEVICE), lengths)).cpu().numpy()
    return pd.DataFrame(rows, index=texts).round(3)

examples = ["This movie was absolutely wonderful, the acting was superb and I loved every minute.",
            "A complete waste of time. The plot made no sense and the acting was terrible.",
            "I expected this to be bad, but it was actually not bad at all.",
            "The first half was great, but the ending ruined the whole thing.",
            "Not good."]
predict_all(examples)

### Thảo luận

Bảng so sánh ở trên xếp hạng các mô hình theo accuracy trên tập test. Các mô hình học máy truyền thống cho kết quả giống nhau trên mọi máy:
**Linear SVM 91,6%**, **Logistic Regression 91,6%**, **Naive Bayes 89,3%**. Kết quả của các mạng nơ-ron có thể dao động vài phần mười phần trăm giữa các lần chạy và giữa CPU/GPU.

**1. Các mô hình tuyến tính trên TF-IDF rất mạnh.** TF-IDF + Linear SVM / Logistic Regression đạt accuracy cao nhất (91,6%) chỉ sau vài giây huấn luyện, trong khi BiGRU cần thời gian huấn luyện lâu hơn nhiều và chỉ đạt khoảng 90–91%.
Trên IMDB, cảm xúc chủ yếu được thể hiện qua **các tín hiệu từ vựng mạnh** (*"worst"*, *"waste"*, *"excellent"*, *"7/10"*; xem bảng hệ số ở mục 4). Một mô hình bag-of-words gồm unigram + bigram, được điều chuẩn tốt và xét *toàn bộ* bài đánh giá, nắm bắt được phần lớn tín hiệu đó.

**2. Vì sao BiGRU không vượt lên:**
* **Cắt ngắn văn bản.** BiGRU chỉ đọc 256 token cuối, nên khoảng 29% bài đánh giá bị cắt, trong khi TF-IDF xét toàn bộ bài. Bảng theo độ dài cho thấy khoảng cách giữa BiGRU và mô hình tuyến tính thường lớn nhất ở nhóm bài dài nhất (> 400 từ).
* **Embedding được học từ đầu** chỉ trên ~40k bài đánh giá. Word vector huấn luyện sẵn (GloVe / word2vec) hoặc transformer huấn luyện sẵn (BERT đạt khoảng 94–95% trên IMDB) mới là thứ thường giúp mô hình chuỗi vượt trội rõ rệt.
* **Mô hình nhỏ:** 64 đơn vị ẩn mỗi chiều, một lớp. Tăng kích thước (128–256 đơn vị, 2 lớp) trên GPU có thể cải thiện thêm.

**3. Vì sao chọn BiGRU làm mô hình RNN.** Trong thử nghiệm sơ bộ với cùng dữ liệu và cùng thiết lập (8 epoch trên CPU), RNN cơ bản (`tanh`) chỉ đạt 83,5%, LSTM một chiều đạt 85,0%, còn BiGRU đạt 90,0%.
RNN cơ bản bị ảnh hưởng nặng bởi hiện tượng tiêu biến gradient trên chuỗi dài (accuracy của nó thấp nhất ở các bài dài nhất). Các cổng của GRU và việc đọc theo hai chiều khắc phục được điều này, với số tham số gần như tương đương.

**4. Mạng nơ-ron cơ bản so với RNN.** Mạng **EmbeddingBag** dùng đúng các token và bộ từ vựng như BiGRU nhưng hoàn toàn bỏ qua thứ tự từ, và đạt accuracy gần như ngang BiGRU với thời gian huấn luyện ngắn hơn nhiều.
Như vậy, *trong thiết lập này*, việc mô hình hóa thứ tự từ chỉ đem lại rất ít so với việc lấy trung bình word embedding. **MLP trên TF-IDF** xấp xỉ Logistic Regression: thêm lớp ẩn lên trên các đặc trưng thưa vốn đã giàu thông tin không giúp được nhiều, và mô hình nhanh chóng bị quá khớp (overfitting) sau 1–2 epoch (loss trên validation tăng trong khi loss trên train tiếp tục giảm, xem đường cong huấn luyện).

**5. Câu có cấu trúc đánh lừa vẫn khó.** Hãy xem bảng các câu tự viết: câu *"I expected this to be bad, but it was actually not bad at all"* (Tôi tưởng phim sẽ dở, nhưng thực ra không dở chút nào) thường bị các mô hình chấm là **tiêu cực**, vì từ *bad* xuất hiện hai lần chiếm ưu thế, và 40k bài đánh giá không đủ để học một cách đáng tin cậy sự đảo nghĩa của "not bad".

**6. Các mô hình mắc lỗi khác nhau.** Có hàng trăm bài trong tập test mà BiGRU đoán đúng còn mô hình truyền thống tốt nhất đoán sai, và ngược lại. Kết hợp (ensemble) xác suất của Logistic Regression và BiGRU thường cho kết quả tốt hơn BiGRU đứng riêng.

**Kết luận thực tiễn.**
| Nếu bạn cần… | Hãy dùng |
|---|---|
| một mốc so sánh mạnh, nhanh, dễ diễn giải | TF-IDF + Linear SVM / Logistic Regression (vài giây trên CPU) |
| một mô hình chuỗi | RNN có cổng (GRU/LSTM), tốt nhất là hai chiều, với embedding huấn luyện sẵn và GPU; không dùng RNN cơ bản cho văn bản dài |
| độ chính xác cao nhất có thể | một transformer huấn luyện sẵn, được tinh chỉnh (fine-tune) |